# Adapting AlphaGenome to MPRA data (Colab)

Minimal example: train the MPRA head on a **frozen** AlphaGenome encoder for one LentiMPRA cell line (HepG2) from [Agarwal et al. 2025](https://www.nature.com/articles/s41586-024-08430-9). Based on `finetune_mpra.py` in [alphagenome_FT_MPRA](https://github.com/Al-Murphy/alphagenome_FT_MPRA).

**Requirements:** GPU runtime.

> **Important:** You will need a **Colab Pro** (or Pro+) account to get GPUs/TPUs with enough VRAM. An **A100 (40 GB)** or better is recommended; the default T4 on free Colab often hits memory limits.

## 1. Install dependencies

Core stack: **alphagenome_ft** (pip) then **alphagenome** and **alphagenome_research** (git). Training data (HepG2 LentiMPRA) is downloaded from [human_legnet](https://github.com/autosome-ru/human_legnet) in the next section.

In [ ]:
#Installation will take a few minutes
# Core stack: alphagenome_ft from pip, then AlphaGenome from git
!pip install -q "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q alphagenome-ft
!pip install -q git+https://github.com/google-deepmind/alphagenome.git
!pip install -q git+https://github.com/google-deepmind/alphagenome_research.git
print("Done.")

In [ ]:
#ensure a GPU is available
import jax
print(jax.devices())  # should show a GPU device

### Helper functions
**MPRA helper code (collapsed).** The cell below defines the classes and functions used in this notebook—`EncoderMPRAHead`, `LentiMPRADataset`, `MPRADataLoader`, and the `train` / `validate` / `test` loops—copied from [alphagenome_FT_MPRA](https://github.com/Al-Murphy/alphagenome_FT_MPRA) so the notebook is self-contained. You can leave it collapsed and run it once; expand only if you want to read or edit the implementation.

In [ ]:
# MPRA helpers (from alphagenome_FT_MPRA: https://github.com/Al-Murphy/alphagenome_FT_MPRA)
from __future__ import annotations
import os
from pathlib import Path
from typing import Any
import jax
import jax.numpy as jnp
import haiku as hk
import pandas as pd
from alphagenome_ft import CustomHead
from alphagenome_research.model import dna_model, layers
try:
    import optax
except ImportError:
    optax = None



#dataset for lentiMPRA data
class LentiMPRADataset:
    def __init__(self, model, path_to_data="./data/legnet_lentimpra", cell_type="HepG2", split="train", test_fold=None, val_fold=None, train_fold=None, promoter_seq="TCCATTATATACCCTCTAGTGTCGGTTCACGCAATG", rand_barcode="AGAGACTGAGGCCAC", organism=None, random_shift=False, random_shift_likelihood=0.5, max_shift=15, reverse_complement=False, reverse_complement_likelihood=0.5, rng_key=None, subset_frac=1.0, pad_n_bases=0):
        if organism is None:
            organism = dna_model.Organism.HOMO_SAPIENS
        test_fold = test_fold or [10]
        val_fold = val_fold or [1]
        train_fold = train_fold or [2, 3, 4, 5, 6, 7, 8, 9]
        assert split in ["train", "val", "test"] and cell_type in ["HepG2", "K562", "WTC11"]
        assert os.path.exists(path_to_data), f"path_to_data must exist: {path_to_data}"
        self.path_to_data, self.cell_type, self.split = path_to_data, cell_type, split
        self.chosen_fold = train_fold if split == "train" else (val_fold if split == "val" else test_fold)
        self.promoter_seq, self.rand_barcode, self.model = promoter_seq, rand_barcode, model
        self.organism = organism
        self.reverse_complement = reverse_complement
        self.reverse_complement_likelihood = reverse_complement_likelihood
        self.random_shift, self.random_shift_likelihood, self.max_shift = random_shift, random_shift_likelihood, max_shift
        self.subset_frac, self.pad_n_bases = subset_frac, pad_n_bases
        self.rng_key = rng_key if rng_key is not None else jax.random.PRNGKey(42)
        self.data = pd.read_csv(os.path.join(path_to_data, f"{cell_type}.tsv"), sep="\t")
        self.data = self.data[self.data["rev"] == 0]
        self.data = self.data[self.data["fold"].isin(self.chosen_fold)].reset_index(drop=True)
        if self.subset_frac < 1.0:
            self.data = self.data.sample(frac=self.subset_frac)
        print(f"Loaded {len(self.data)} samples for {split} split")
    def __len__(self):
        return len(self.data)
    def _reverse_complement_onehot(self, seq_onehot, force=False):
        if not force:
            self.rng_key, subkey = jax.random.split(self.rng_key)
            if jax.random.uniform(subkey, ()) > self.reverse_complement_likelihood:
                return seq_onehot, self.rng_key
        strand_reindexing = jnp.array([3, 2, 1, 0])
        if seq_onehot.ndim == 2:
            rev_comp = seq_onehot[::-1, :][:, strand_reindexing]
        else:
            rev_comp = seq_onehot[:, ::-1, :][:, :, strand_reindexing]
        return rev_comp, self.rng_key
    def _random_shift_onehot(self, seq_onehot, force=False):
        if not force:
            self.rng_key, subkey = jax.random.split(self.rng_key)
            if jax.random.uniform(subkey, ()) > self.random_shift_likelihood:
                return seq_onehot, self.rng_key
        self.rng_key, subkey = jax.random.split(self.rng_key)
        shift = jax.random.randint(subkey, (), -self.max_shift, self.max_shift + 1)
        return jnp.roll(seq_onehot, shift, axis=0), self.rng_key
    def __getitem__(self, idx):
        organism_index = jnp.array([0]) if self.organism == dna_model.Organism.HOMO_SAPIENS else jnp.array([1])
        sequence = self.data.iloc[idx]['seq'] + self.promoter_seq + self.rand_barcode
        if self.pad_n_bases > 0:
            p = self.pad_n_bases // 2
            sequence = 'N' * p + sequence + 'N' * p
        label = self.data.iloc[idx]['mean_value']
        sequence_onehot = jnp.array(self.model._one_hot_encoder.encode(sequence))
        if self.reverse_complement:
            sequence_onehot, self.rng_key = self._reverse_complement_onehot(sequence_onehot)
        if self.random_shift:
            sequence_onehot, self.rng_key = self._random_shift_onehot(sequence_onehot)
        return {"seq": sequence_onehot, "y": label, "organism_index": organism_index}

#dataloader for lentiMPRA data
class MPRADataLoader:
    def __init__(self, dataset, batch_size=32, shuffle=False, rng_key=None):
        self.dataset, self.batch_size, self.shuffle = dataset, batch_size, shuffle
        self.rng_key = rng_key if rng_key is not None else (jax.random.PRNGKey(42) if shuffle else None)
    def __iter__(self):
        indices = jnp.arange(len(self.dataset))
        if self.shuffle:
            self.rng_key, subkey = jax.random.split(self.rng_key)
            indices = jax.random.permutation(subkey, indices)
        num_batches = (len(self.dataset) + self.batch_size - 1) // self.batch_size
        for i in range(num_batches):
            start = i * self.batch_size
            end = min(start + self.batch_size, len(self.dataset))
            batch_samples = [self.dataset[int(k)] for k in indices[start:end].tolist()]
            yield self._stack_batch(batch_samples)
    def _stack_batch(self, samples):
        if "encoder_output" in samples[0]:
            encoder_outputs = [s["encoder_output"] for s in samples]
            max_len, max_feat = max(e.shape[0] for e in encoder_outputs), encoder_outputs[0].shape[1]
            padded = [jnp.concatenate([e, jnp.zeros((max_len - e.shape[0], max_feat))], axis=0) if e.shape[0] < max_len else e for e in encoder_outputs]
            return {"encoder_output": jnp.stack(padded), "y": jnp.array([s["y"] for s in samples]), "organism_index": jnp.stack([s["organism_index"] for s in samples]).squeeze(-1)}
        seqs = [s["seq"] for s in samples]
        max_len = max(s.shape[0] for s in seqs)
        padded = [jnp.concatenate([s, jnp.zeros((max_len - s.shape[0], 4))], axis=0) if s.shape[0] < max_len else s for s in seqs]
        return {"seq": jnp.stack(padded), "y": jnp.array([s["y"] for s in samples]), "organism_index": jnp.stack([s["organism_index"] for s in samples]).squeeze(-1)}
    def __len__(self):
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size


#simple training/valid/test loop scripts
def validate(model, dataloader, loss_fn, head_name="mpra_head"):
    total_loss, total_pearson, num_batches = 0.0, 0.0, 0
    strand_reindex = jax.device_put(model._metadata[dna_model.Organism.HOMO_SAPIENS].strand_reindexing, model._device_context._device)
    for batch in dataloader:
        with model._device_context:
            predictions = model._predict(model._params, model._state, batch["seq"], batch["organism_index"], negative_strand_mask=jnp.zeros(batch["seq"].shape[0], dtype=bool), strand_reindexing=strand_reindex)
        loss_dict = loss_fn(predictions[head_name], {"targets": batch["y"]})
        total_loss += float(loss_dict["loss"])
        total_pearson += float(loss_dict.get("pearson_corr", 0.0))
        num_batches += 1
    n = num_batches or 1
    return {"val_loss": total_loss / n, "val_pearson": total_pearson / n}

def test(model, dataloader, loss_fn, head_name="mpra_head"):
    total_loss, total_pearson, num_batches = 0.0, 0.0, 0
    strand_reindex = jax.device_put(model._metadata[dna_model.Organism.HOMO_SAPIENS].strand_reindexing, model._device_context._device)
    for batch in dataloader:
        with model._device_context:
            predictions = model._predict(model._params, model._state, batch["seq"], batch["organism_index"], negative_strand_mask=jnp.zeros(batch["seq"].shape[0], dtype=bool), strand_reindexing=strand_reindex)
        loss_dict = loss_fn(predictions[head_name], {"targets": batch["y"]})
        total_loss += float(loss_dict["loss"])
        total_pearson += float(loss_dict.get("pearson_corr", 0.0))
        num_batches += 1
    n = num_batches or 1
    return {"test_loss": total_loss / n, "test_pearson": total_pearson / n}

def train(model, train_loader, val_loader=None, test_loader=None, num_epochs=10, learning_rate=1e-3, head_name="mpra_head", checkpoint_dir=None, save_minimal_model=True, early_stopping_patience=5, use_wandb=False, **kwargs):
    if optax is None:
        raise ImportError("optax is required. pip install optax")
    rng_key = jax.random.PRNGKey(42)
    loss_fn = model.create_loss_fn_for_head(head_name)
    optimizer = optax.adam(learning_rate)
    opt_state = optimizer.init(model._params)
    strand_reindex = jax.device_put(model._metadata[dna_model.Organism.HOMO_SAPIENS].strand_reindexing, model._device_context._device)
    history = {"train_loss": [], "train_pearson": [], "val_loss": [], "val_pearson": [], "test_loss": [], "test_pearson": []}
    best_val_loss, patience_counter = float("inf"), 0
    print(f"Starting training for {num_epochs} epochs...", flush=True)
    for epoch in range(num_epochs):
        train_losses = []
        for batch in train_loader:
            rng_key, _ = jax.random.split(rng_key)
            seq_batch, targets, org_idx = batch["seq"], batch["y"], batch["organism_index"]
            def loss_fn_inner(params):
                with model._device_context:
                    preds = model._predict(params, model._state, seq_batch, org_idx, negative_strand_mask=jnp.zeros(seq_batch.shape[0], dtype=bool), strand_reindexing=strand_reindex)
                return loss_fn(preds[head_name], {"targets": targets})["loss"]
            loss, grads = jax.value_and_grad(loss_fn_inner)(model._params)
            updates, opt_state = optimizer.update(grads, opt_state, model._params)
            model._params = optax.apply_updates(model._params, updates)
            train_losses.append(float(loss))
        avg_train_loss = sum(train_losses) / len(train_losses) if train_losses else 0.0
        train_metrics = validate(model, train_loader, loss_fn, head_name)
        history["train_loss"].append(avg_train_loss)
        history["train_pearson"].append(train_metrics["val_pearson"])
        val_metrics = validate(model, val_loader, loss_fn, head_name) if val_loader else {"val_loss": float("inf"), "val_pearson": 0.0}
        history["val_loss"].append(val_metrics["val_loss"])
        history["val_pearson"].append(val_metrics["val_pearson"])
        if test_loader:
            t = test(model, test_loader, loss_fn, head_name)
            history["test_loss"].append(t["test_loss"])
            history["test_pearson"].append(t["test_pearson"])
        print(f"Epoch {epoch + 1}: train_loss={avg_train_loss:.6f}, train_pearson={train_metrics['val_pearson']:.4f}, val_loss={val_metrics['val_loss']:.6f}, val_pearson={val_metrics['val_pearson']:.4f}", flush=True)
        if val_metrics["val_loss"] < best_val_loss:
            best_val_loss, patience_counter = val_metrics["val_loss"], 0
            if checkpoint_dir:
                Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
                model.save_checkpoint(checkpoint_dir, save_full_model=False, save_minimal_model=save_minimal_model)
        else:
            patience_counter += 1
            if early_stopping_patience and patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch + 1}", flush=True)
                break
    return history

## 2. Data

HepG2 LentiMPRA training data is downloaded from [human_legnet](https://github.com/autosome-ru/human_legnet) (`datasets/original/HepG2.tsv`).

In [ ]:
import os
import urllib.request
import pandas as pd

# Download HepG2 LentiMPRA data from human_legnet (same format as LentiMPRADataset expects)
DATA_DIR = "/tmp/legnet_lentimpra"
os.makedirs(DATA_DIR, exist_ok=True)
HEPG2_URL = "https://raw.githubusercontent.com/autosome-ru/human_legnet/main/datasets/original/HepG2.tsv"
hepg2_path = os.path.join(DATA_DIR, "HepG2.tsv")
urllib.request.urlretrieve(HEPG2_URL, hepg2_path)
print("DATA_DIR:", DATA_DIR)

# First few lines of training data (HepG2; train split uses folds 2–9)
df = pd.read_csv(hepg2_path, sep="\t")
df = df[df["rev"] == 0]  # same filter as LentiMPRADataset
train_df = df[df["fold"].isin([2, 3, 4, 5, 6, 7, 8, 9])]
print(f"\nTraining samples: {len(train_df)}. First few rows:")
train_df.head()

## 3. Imports and config

In [ ]:
from pathlib import Path
from alphagenome.models import dna_output
from alphagenome_ft import HeadConfig, HeadType, register_custom_head, create_model_with_custom_heads
# EncoderMPRAHead, LentiMPRADataset, MPRADataLoader, train defined in the collapsed cell above

# Stage 1 (frozen backbone) config from mpra_HepG2.json
CELL_TYPE = "HepG2"
# Faster demo: 10% of data, batch 16. For full training set SUBSET_FRAC=1.0 and BATCH_SIZE=32.
SUBSET_FRAC = 0.1
BATCH_SIZE = 16
NUM_EPOCHS = 5  # increase (e.g. 100) for full training
LEARNING_RATE = 0.001
RANDOM_SHIFT_LIKELIHOOD = 0.5
PAD_N_BASES = 0
EARLY_STOPPING_PATIENCE = 5
# Head: center_bp=256, pooling_type="flatten", nl_size=1024, do=0.1, activation="relu"

CHECKPOINT_DIR = "/content/checkpoints_mpra"
PROMOTER_CONSTRUCT_LENGTH = 281
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

## 4. Model: create/register head, create, freeze backbone

In [ ]:
import jax.numpy as jnp
import haiku as hk
from alphagenome.models import dna_output
from alphagenome_research.model import layers

#custom head for MPRA - need to define a predict & loss function
class EncoderMPRAHead(CustomHead):
    #predefined by AlphaGenome
    ENCODER_RESOLUTION_BP = 128
    
    def __init__(self, *, name, output_type, num_tracks, num_organisms, metadata):
        super().__init__(name=name, num_tracks=num_tracks, output_type=output_type, num_organisms=num_organisms, metadata=metadata)
        center_bp = metadata.get('center_bp', 256) if metadata else 256
        nl_size = metadata.get('nl_size', 1024) if metadata else 1024
        self._hidden_sizes = [nl_size] if isinstance(nl_size, int) else nl_size
        self._do = metadata.get('do', None) if metadata else None
        self._center_window_positions = max(1, int(center_bp / self.ENCODER_RESOLUTION_BP))
        self._pooling_type = metadata.get('pooling_type', 'sum') if metadata else 'sum'
        assert self._pooling_type in ['sum', 'mean', 'max', 'center', 'flatten']
        self._activation = metadata.get('activation', 'relu') if metadata else 'relu'
        assert self._activation in ['relu', 'gelu']
    
    def predict(self, embeddings, organism_index, **kwargs):
        if not hasattr(embeddings, 'encoder_output') or embeddings.encoder_output is None:
            raise AttributeError("EncoderMPRAHead requires encoder_output. Use use_encoder_output=True.")
        x = embeddings.encoder_output
        #layer norm helped speed up convergence
        x = layers.LayerNorm(name='norm')(x)
        
        if self._pooling_type == 'flatten':
            x = x.reshape(x.shape[0], -1)
        
        for i, hidden_size in enumerate(self._hidden_sizes):
            x = hk.Linear(hidden_size, name=f'hidden_{i}')(x)
            if self._do is not None:
                try:
                    x = hk.dropout(hk.next_rng_key(), self._do, x)
                except (RuntimeError, ValueError, AttributeError):
                    pass
            x = jax.nn.gelu(x) if self._activation == 'gelu' else jax.nn.relu(x)
        
        if self._pooling_type == 'flatten':
            per_position_predictions = hk.Linear(self._num_tracks, name='output')(x)[:, None, :]
        else:
            per_position_predictions = hk.Linear(self._num_tracks, name='output')(x)
        return per_position_predictions
    
    def loss(self, predictions, batch):
        targets = batch.get('targets')
        if targets is None:
            return {'loss': jnp.array(0.0)}
        seq_len = predictions.shape[1]
        if self._pooling_type == 'flatten':
            pred_values = predictions.squeeze(1)
        elif self._pooling_type == 'center':
            center_idx = seq_len // 2
            pred_values = jax.lax.dynamic_slice_in_dim(predictions, center_idx, 1, axis=1).squeeze(1)
        else:
            window_size = min(int(self._center_window_positions), seq_len)
            center_start = max((seq_len - window_size) // 2, 0)
            center_predictions = jax.lax.dynamic_slice_in_dim(predictions, center_start, window_size, axis=1)
            if self._pooling_type == 'mean':
                pred_values = jnp.mean(center_predictions, axis=1)
            elif self._pooling_type == 'max':
                pred_values = jnp.max(center_predictions, axis=1)
            else:
                pred_values = jnp.sum(center_predictions, axis=1)
        if targets.ndim == 1:
            targets = targets[:, None]
        mse_loss = jnp.mean((pred_values - targets) ** 2)
        pred_flat, targets_flat = pred_values.flatten(), targets.flatten()
        pearson_corr = jnp.corrcoef(pred_flat, targets_flat)[0, 1]
        
        return {'loss': mse_loss, 'mse': mse_loss, 'pearson_corr': pearson_corr}

In [ ]:
#You first register a new head
##reusing the predefined RNA_SEQ head type from `alphagenome_research`
##set number of output tracks to 1 for single task predictions
##these hyperparameters match those used for **Stage 1 (frozen backbone)** fine-tuning our analysis
register_custom_head(
    "mpra_head",
    EncoderMPRAHead,
    HeadConfig(
        type=HeadType.GENOME_TRACKS,
        output_type=dna_output.OutputType.RNA_SEQ,
        num_tracks=1,
        metadata={
            "center_bp": 256,
            "pooling_type": "flatten",
            "nl_size": 1024,
            "do": 0.1,
            "activation": "relu",
        },
    ),
)

### Kaggle credentials (required for downloading AlphaGenome weights)

The next cell downloads the base Alphagenome model weights from Kaggle when creating the encoder fine-tuning model. 

**One-time setup:**

1. **Accept the dataset agreement:** [AlphaGenome on Kaggle](https://www.kaggle.com/datasets/google-deepmind/alphagenome) → open the page and accept terms if prompted.
2. **Create a _legacy_ API token:** [Kaggle Account → Create New Token](https://www.kaggle.com/settings) (downloads `kaggle.json`). Use the **username** and **key** from that file (the **key** is the long token, not your account password). 

**NOTE** - do not use a 'API Tokens (Recommended)' as this will **not** work.

Once you log in and it's successful, rerun the code to create the model.

In [ ]:
# Create the AlphaGenome model (encoder + custom head only).
model = create_model_with_custom_heads(
    "all_folds", custom_heads=["mpra_head"], use_encoder_output=True, init_seq_len=PROMOTER_CONSTRUCT_LENGTH
)
model.freeze_except_head("mpra_head")
print("Model ready.")

## 5. Datasets and dataloaders

**Note:** The config above uses **10% of the data** (`SUBSET_FRAC=0.1`) and **batch size 16** so the notebook runs in a reasonable time. For full training, set `SUBSET_FRAC=1.0` and `BATCH_SIZE=32` in the previous cell and re-run from there.

In [ ]:
train_dataset = LentiMPRADataset(
    model=model, path_to_data=DATA_DIR, cell_type=CELL_TYPE, split="train",
    random_shift=True, random_shift_likelihood=RANDOM_SHIFT_LIKELIHOOD,
    reverse_complement=True, pad_n_bases=PAD_N_BASES, subset_frac=SUBSET_FRAC,
)
val_dataset = LentiMPRADataset(
    model=model, path_to_data=DATA_DIR, cell_type=CELL_TYPE, split="val",
    random_shift=False, reverse_complement=False, pad_n_bases=PAD_N_BASES, subset_frac=SUBSET_FRAC,
)
test_dataset = LentiMPRADataset(
    model=model, path_to_data=DATA_DIR, cell_type=CELL_TYPE, split="test",
    random_shift=False, reverse_complement=False, pad_n_bases=PAD_N_BASES, subset_frac=SUBSET_FRAC,
)
train_loader = MPRADataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = MPRADataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = MPRADataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("Train:", len(train_dataset), "Val:", len(val_dataset), "Test:", len(test_dataset))

## 6. Train

In [ ]:
history = train(
    model, train_loader, val_loader=val_loader, test_loader=test_loader,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE, checkpoint_dir=CHECKPOINT_DIR,
    save_minimal_model=True, early_stopping_patience=EARLY_STOPPING_PATIENCE, use_wandb=False,
)

## 7. Plot loss and Pearson

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(history["train_loss"], label="Train"); ax1.plot(history["val_loss"], label="Val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE"); ax1.legend(); ax1.set_title("Loss")
ax2.plot(history["train_pearson"], label="Train"); ax2.plot(history["val_pearson"], label="Val")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Pearson r"); ax2.legend(); ax2.set_title("Pearson")
plt.tight_layout(); plt.show()
if history["test_pearson"]:
    print("Final test Pearson:", round(history["test_pearson"][-1], 4))